In [1]:
# ==============================
# Federated Learning on EMNIST Letters (Dirichlet Non-IID)
# Model: LeNet++ CNN
# Algorithm: FedSGD + Masking + Adaptive Noise
# Tracks: Train/Val/Test Loss & Accuracy + Round Time + Total Time
# ==============================

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
import random
import copy
from collections import OrderedDict
import time

# -------------------------
# Configuration
# -------------------------
NUM_CLIENTS = 100
CLIENT_FRAC = 0.3
ROUNDS = 60
LOCAL_EPOCHS = 3
BATCH_SIZE = 128
LR = 0.01
MU = 0.01
MASK_SCALE = 0.001
ADAPTIVE_NOISE_FACTOR = 0.2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
ALPHA = 0.1

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.benchmark = True

print("Device:", DEVICE)

# -------------------------
# Dataset: EMNIST Letters
# -------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.EMNIST(
    root="./data",
    split="letters",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.EMNIST(
    root="./data",
    split="letters",
    train=False,
    download=True,
    transform=transform
)

# EMNIST Letters: 26 classes
NUM_CLASSES = 26

# IMPORTANT: labels are 1–26 → convert to 0–25
train_dataset.targets = train_dataset.targets - 1
test_dataset.targets = test_dataset.targets - 1

# -------------------------
# Validation Split
# -------------------------
VAL_CLIENTS = list(range(10))
TRAIN_CLIENTS = list(range(10, NUM_CLIENTS))

# -------------------------
# Dirichlet Non-IID Partition
# -------------------------
def dirichlet_partition(dataset, clients, alpha=0.1):
    labels = np.array(dataset.targets)
    idxs = np.arange(len(labels))
    client_dict = {i: [] for i in clients}

    class_indices = [idxs[labels == i] for i in range(NUM_CLASSES)]

    for c_idx in class_indices:
        np.random.shuffle(c_idx)
        proportions = np.random.dirichlet(alpha * np.ones(len(clients)))
        proportions = (np.cumsum(proportions) * len(c_idx)).astype(int)
        proportions = np.concatenate(([0], proportions))

        for i, client in enumerate(clients):
            client_dict[client] += list(c_idx[proportions[i]:proportions[i+1]])

    for client in clients:
        np.random.shuffle(client_dict[client])

    return client_dict

client_data = dirichlet_partition(train_dataset, list(range(NUM_CLIENTS)), alpha=ALPHA)

# -------------------------
# Model: CNN (LeNet++)
# -------------------------
class EMNIST_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(64*7*7, 256)
        self.fc2 = nn.Linear(256, NUM_CLASSES)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

# -------------------------
# Vector Utils
# -------------------------
def model_to_vector(model):
    return torch.cat([p.data.view(-1) for p in model.parameters()])

def vector_to_model(model, vec):
    pointer = 0
    for p in model.parameters():
        num = p.numel()
        p.data = vec[pointer:pointer+num].view_as(p).clone()
        pointer += num

# -------------------------
# Evaluation
# -------------------------
def evaluate(model, loader):
    model.eval()
    correct, total, loss_val = 0, 0, 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss_val += F.cross_entropy(out, y).item() * y.size(0)
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return 100 * correct / total, loss_val / total

# -------------------------
# Training
# -------------------------
global_model = EMNIST_CNN().to(DEVICE)
global_vec = model_to_vector(global_model)

train_acc, val_acc, test_acc = [], [], []
loss_curve, train_loss_curve, val_loss_curve, test_loss_curve = [], [], [], []
round_time, total_time = [], []

start_total = time.time()

for rnd in range(ROUNDS):
    start_round = time.time()

    selected_clients = random.sample(TRAIN_CLIENTS, int(NUM_CLIENTS*CLIENT_FRAC))

    masks = {cid: torch.zeros_like(global_vec) for cid in selected_clients}
    sorted_ids = sorted(selected_clients)
    K = len(sorted_ids)

    for idx, cid in enumerate(sorted_ids):
        left = sorted_ids[(idx - 1) % K]
        right = sorted_ids[(idx + 1) % K]

        if cid < right:
            rmask = torch.randn_like(global_vec) * MASK_SCALE
            masks[cid] += rmask
            masks[right] -= rmask

    updates, local_losses = [], []

    for cid in selected_clients:
        local_model = copy.deepcopy(global_model)
        local_model.train()

        optimizer = optim.SGD(local_model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)

        loader = DataLoader(
            Subset(train_dataset, client_data[cid]),
            batch_size=BATCH_SIZE,
            shuffle=True
        )

        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            optimizer.zero_grad()

            out = local_model(x)
            loss = F.cross_entropy(out, y)

            loss.backward()
            optimizer.step()

        local_losses.append(loss.item())

        delta = model_to_vector(local_model) - global_vec

        pointer = 0
        for i, p in enumerate(local_model.parameters()):
            num = p.numel()
            layer_update = delta[pointer:pointer+num].view_as(p)
            layer_norm = torch.norm(layer_update)

            noise = torch.randn_like(layer_update) * layer_norm * ADAPTIVE_NOISE_FACTOR
            delta[pointer:pointer+num] += noise.view(-1)

            pointer += num

        delta += masks[cid]
        updates.append(delta)

    avg_update = torch.stack(updates).mean(0)
    global_vec += avg_update
    vector_to_model(global_model, global_vec)

    # -------------------------
    # Metrics
    # -------------------------
    train_loader = DataLoader(
        torch.utils.data.ConcatDataset([
            Subset(train_dataset, client_data[c]) for c in selected_clients
        ]),
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    val_loader = DataLoader(
        torch.utils.data.ConcatDataset([
            Subset(train_dataset, client_data[c]) for c in VAL_CLIENTS
        ]),
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    tr_acc, tr_loss = evaluate(global_model, train_loader)
    va_acc, va_loss = evaluate(global_model, val_loader)
    te_acc, te_loss = evaluate(global_model, test_loader)

    train_acc.append(tr_acc)
    val_acc.append(va_acc)
    test_acc.append(te_acc)

    loss_curve.append(np.mean(local_losses))
    train_loss_curve.append(tr_loss)
    val_loss_curve.append(va_loss)
    test_loss_curve.append(te_loss)

    rt = time.time() - start_round
    round_time.append(rt)
    total_time.append(time.time() - start_total)

    print(f"Round {rnd+1:03d} | "
          f"LocalLoss {loss_curve[-1]:.4f} | "
          f"TrainLoss {tr_loss:.4f} | "
          f"ValLoss {va_loss:.4f} | "
          f"TestLoss {te_loss:.4f} | "
          f"Train {tr_acc:.4f} | Val {va_acc:.4f} | Test {te_acc:.4f} | "
          f"RoundTime {rt:.2f}s | TotalTime {total_time[-1]:.2f}s")

Device: cuda
Round 001 | LocalLoss 2.0380 | TrainLoss 3.2166 | ValLoss 3.2491 | TestLoss 3.2608 | Train 5.9783 | Val 14.8837 | Test 6.6538 | RoundTime 26.76s | TotalTime 26.76s
Round 002 | LocalLoss 1.3781 | TrainLoss 3.1155 | ValLoss 3.2187 | TestLoss 3.2154 | Train 14.0010 | Val 4.6221 | Test 9.9087 | RoundTime 27.95s | TotalTime 54.71s
Round 003 | LocalLoss 1.2637 | TrainLoss 3.0084 | ValLoss 3.1076 | TestLoss 3.0935 | Train 14.7805 | Val 10.4288 | Test 12.3125 | RoundTime 25.78s | TotalTime 80.49s
Round 004 | LocalLoss 1.3663 | TrainLoss 2.9014 | ValLoss 2.9779 | TestLoss 2.9353 | Train 15.3168 | Val 15.1672 | Test 16.3894 | RoundTime 22.17s | TotalTime 102.66s
Round 005 | LocalLoss 0.9566 | TrainLoss 2.7386 | ValLoss 2.7321 | TestLoss 2.8333 | Train 22.5183 | Val 25.2544 | Test 20.5769 | RoundTime 31.83s | TotalTime 134.49s
Round 006 | LocalLoss 0.8353 | TrainLoss 2.6142 | ValLoss 2.6589 | TestLoss 2.6106 | Train 25.9293 | Val 26.4390 | Test 25.5433 | RoundTime 24.55s | TotalTime 

In [ ]:
# -------------------------
# Save metrics to CSV
# -------------------------
import csv

metrics = {
    "round": list(range(1, ROUNDS+1)),
    "train_acc": train_acc,
    "val_acc": val_acc,
    "test_acc": test_acc,
    "local_loss": loss_curve,
    "train_loss": train_loss_curve,
    "val_loss": val_loss_curve,
    "test_loss": test_loss_curve,
    "round_time": round_time,
    "total_time": total_time
}

with open("EMNIST_Letters_masking_federated_metrics_2_fedsgd.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(metrics.keys())
    writer.writerows(zip(*metrics.values()))

# Colab download
try:
    from google.colab import files
    files.download("EMNIST_Letters_masking_federated_metrics_2_fedsgd.csv")
except:
    print("CSV saved locally as EMNIST_Letters_masking_federated_metrics_2_fedsgd.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>